# 03 · Fama-French 5 Factor Model
**Professional-grade alpha decomposition**

This notebook implements the buy-side standard attribution model:

$$R_p - R_f = \\alpha + \\beta_1(\\text{Mkt-RF}) + \\beta_2(\\text{SMB}) + \\beta_3(\\text{HML}) + \\beta_4(\\text{RMW}) + \\beta_5(\\text{CMA}) + \\varepsilon$$

**Factor meaning:**
| Factor | Meaning | You want |
|--------|---------|----------|
| Mkt-RF | Systematic market risk | β ≈ 0.9–1.1 for target beta |
| SMB    | Size premium (small-cap) | ≈0 (you hold large-caps) |
| HML    | Value premium (high B/M) | depends on style |
| RMW    | Profitability premium | > 0 (quality tilt) |
| CMA    | Conservative investment | > 0 (disciplined capex) |
| α      | **Skill — risk-adjusted excess return** | > 0 |

Data: Kenneth French Data Library (academic standard, free)

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 130, "font.size": 11})
print("Libraries loaded ✓")

## 1 · Load Portfolio Returns

In [ ]:
from src.data_fetcher import build_portfolio_snapshot, get_historical_prices
from src.portfolio    import compute_portfolio_returns, compute_weights

snap     = build_portfolio_snapshot(use_sheets=True)
snap     = compute_weights(snap)
tickers  = snap["ticker"].tolist()

print("Fetching price history …")
prices   = get_historical_prices(tickers, period="2y")
port_ret = compute_portfolio_returns(prices, snap)
print(f"Portfolio returns: {len(port_ret)} observations")
print(f"Date range: {port_ret.index[0].date()} → {port_ret.index[-1].date()}")

## 2 · Fetch FF5 Factors (Kenneth French Library)

In [ ]:
from src.factors import fetch_ff5_factors

start = str(port_ret.index.min().date())
end   = str(port_ret.index.max().date())

print(f"Downloading Fama-French 5 Factors: {start} to {end}")
factors = fetch_ff5_factors(start, end)
print(f"Factors loaded: {len(factors)} observations")
print("\nFactor summary (daily means in bps):")
(factors * 10000).describe().loc[["mean","std"]].T.round(2)

## 3 · OLS Regression — Factor Attribution

In [ ]:
from src.factors import run_factor_regression

reg = run_factor_regression(port_ret, factors)

alpha_sig = "✅ Statistically significant" if abs(reg['alpha_tstat']) > 2.0 else "⚠️  Not significant (t < 2)"

print(f"""
╔══════════════════════════════════════════════════════════════╗
  Fama-French 5 Factor Regression Results
╠══════════════════════════════════════════════════════════════╣
  α (alpha, annualised) :  {reg['alpha_annual_pct']:+.3f}%  {alpha_sig}
  α t-stat              :  {reg['alpha_tstat']:+.3f}   p = {reg['alpha_pval']:.4f}
  R²                    :  {reg['r_squared']:.4f}
  Adj. R²               :  {reg['adj_r_squared']:.4f}
  N observations        :  {reg['n_obs']}
╠══════════════════════════════════════════════════════════════╣
  Factor Betas  (loading / t-stat):""")

factor_cols = list(reg["betas"].keys())
for f in factor_cols:
    b = reg["betas"][f]
    t = reg["tstats"][f]
    p = reg["pvalues"][f]
    sig = "**" if abs(t) > 2.0 else "  "
    print(f"  {f:<10} {b:+.4f}   t={t:+.2f}  {sig}")

print("╚══════════════════════════════════════════════════════════════╝")
print("\n** = statistically significant at 95% confidence")

## 4 · Return Attribution Waterfall

In [ ]:
from src.factors import decompose_returns, attribution_summary

decomp = decompose_returns(reg, factors, port_ret)
attrib = attribution_summary(decomp)

# Visualise annual contribution of each component
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#27ae60" if x >= 0 else "#e74c3c" for x in attrib.annual_contrib_pct]
bars = ax.bar(attrib.component, attrib.annual_contrib_pct, color=colors, edgecolor="white", width=0.6)
ax.axhline(0, color="white", lw=1)
ax.set_ylabel("Annualised Contribution (%)")
ax.set_title("Return Attribution — FF5 Factor Decomposition", fontsize=13)
for bar, val in zip(bars, attrib.annual_contrib_pct):
    ax.text(bar.get_x() + bar.get_width()/2, val + (0.1 if val >= 0 else -0.3),
            f"{val:+.2f}%", ha="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nAttribution detail:")
attrib

## 5 · Rolling Factor Betas (6-month window)

In [ ]:
from src.factors import rolling_factor_betas

roll = rolling_factor_betas(port_ret, factors, window=126)

fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=True)
factor_cols_plot = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "alpha_annual_pct"]
titles = ["Market Beta (Mkt-RF)", "Size Beta (SMB)", "Value Beta (HML)",
          "Profitability Beta (RMW)", "Investment Beta (CMA)", "Rolling Alpha (annual %)"]
colors_f = ["#4C72B0","#DD8452","#2ca02c","#d62728","#9467bd","#17becf"]

for ax, col, title, c in zip(axes.flat, factor_cols_plot, titles, colors_f):
    ax.plot(roll.index, roll[col], color=c, lw=1.5)
    ax.axhline(roll[col].mean(), color="grey", ls="--", lw=1,
               label=f"avg = {roll[col].mean():.3f}")
    ax.axhline(0, color="white", lw=0.8)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle("Rolling 6-Month Factor Exposures", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6 · Individual Position Factor Scores

In [ ]:
from src.factors import run_position_betas

print("Running FF3 regressions on individual positions …")
pos_betas = run_position_betas(prices, factors)

print("\nPosition Factor Scores (sorted by alpha):")
pos_betas

In [ ]:
# Visualise: market beta vs alpha bubble chart
fig, ax = plt.subplots(figsize=(12, 7))
scatter = ax.scatter(
    pos_betas.beta_Mkt,
    pos_betas.alpha_annual_pct,
    s=200,
    c=pos_betas.r_squared,
    cmap="YlOrRd",
    edgecolors="white",
    linewidths=0.8,
    zorder=3,
)
for _, row in pos_betas.iterrows():
    ax.annotate(row.ticker,
                (row.beta_Mkt, row.alpha_annual_pct),
                textcoords="offset points",
                xytext=(6, 4),
                fontsize=9)

ax.axhline(0, color="white", ls="--", lw=1)
ax.axvline(1, color="grey",  ls="--", lw=1, label="β=1")
ax.set_xlabel("Market Beta (Mkt-RF)", fontsize=12)
ax.set_ylabel("Alpha (annualised %)", fontsize=12)
ax.set_title("Position Alpha vs Market Beta  (colour = R²)", fontsize=13)
plt.colorbar(scatter, ax=ax, label="R²")
ax.legend()
plt.tight_layout()
plt.show()

## 7 · Regression Summary (statsmodels full output)

In [ ]:
# Full OLS regression table — what you'd present in a research report
print(reg["model"].summary())